In [ ]:
import numpy as np
import pandas as pd
import folium
import geopandas as gdp
import networkx as nx
import osmnx as ox
from shapely.geometry import Polygon, MultiPolygon


## 1. 設定

In [ ]:
lat_lon_of_target = (35.651809, 139.544677)  #　調布駅。ベットタウンの１つの調布をサンプルとして選択。

dist = 2000 # メートル

network_type = "walk" # 他には、"all_private", "all", "bike", "drive_service", "drive", がある。
walk_speed = 80 # メートル/min

travel_times = [5, 10, 15]  # 分。移動時間。
cutoff_distances = [t * walk_speed for t in travel_times] #　移動速度を距離に変換

## 2. 地図情報の取得

In [ ]:
print("道路ネットワークを取得中")

G = ox.graph_from_point(
    lat_lon_of_target, dist=dist, network_type=network_type, simplify=True
)

print("取得完了")


In [ ]:
# 中心座標に一番近いノード（交差点など）を取得
center_node = ox.distance.nearest_nodes(
    G, X=lat_lon_of_target[1], Y=lat_lon_of_target[0]
)

In [ ]:
subgraphs = []
colors = ["red", "green", "blue"]

# NetworkXのグラフから、中心ノードから到達可能なノードを検索して取得
for i, max_dist in enumerate(cutoff_distances):
  # weight="length"（距離ベース）または時間ベースの重み
  subgraph = nx.ego_graph(
      G, center_node, radius=max_dist, distance="length"
  )
  subgraphs.append(subgraph)

  print(f"{i+1}/{len(cutoff_distances)}")

## 3. マップの可視化

In [ ]:
# 中心ノードからのポリゴン（特定エリアのデータ）を作成してhtml形式で可視化
# 注釈：ポリゴン（Polygon）は、地図上のある特定の「面（エリア・一区画）」を表すデータ形式

m = folium.Map(
    location=lat_lon_of_target,
    zoom_start=15,
    tiles="https://cyberjapandata.gsi.go.jp/xyz/std/{z}/{x}/{y}.png",
    attr="<a href='https://gsi.go.jp' target='_blank'>国土地理院</a>",
)

In [ ]:

# 徒歩圏の距離ごとにポリゴンを作成
for subg, color, t in zip(subgraphs, colors, travel_times):
  # osmnxのグラフをGeoDataFrameに変換
  gdf_node = ox.graph_to_gdfs(subg, edges=False)

  # ノードの座標を結合し、単一のジオメトリを作成。
  # そして、凸包（Convex Hull）を利用し到達圏ポリゴンを作成
  # 凸包（Convex Hull）：空間上にある複数の点（ポイント）や図形をすべて包み込む、最小の凸多角形（へこみのない多角形）
  if len(gdf_node) > 2:
    polygon = gdf_node.unary_union.convex_hull
    
    # Foliumに追加
    folium.GeoJson(
        polygon,
        style_function=lambda x, col=color: {
            "fillColor": col,
            "color": col,
            "weight": 1,
            "fillOpacity": 0.3,
        },
        tooltip=f"徒歩約{t}分圏",
    ).add_to(m)

In [ ]:
# 中心点を出発地点としてfoliumにマーク
folium.Marker(
    lat_lon_of_target,
    popup="出発地",
    icon=folium.Icon(color="black")
).add_to(m)

In [ ]:
m.save("./data/output/map_tmp.html")